<a href="https://colab.research.google.com/github/vikrantraj53036-commits/vikrant-2026B15410292/blob/main/Project24_Mobile_App_Usage_Analysis_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 24: Mobile App Usage & User Engagement Analysis

**Name:** Vikrant Raj  
**Admission Number:** 2026B15410292  
**Industry:** Technology / Mobile Applications  
**Dataset:** `24_mobile_app_usage.csv`

**Business scenario:** study app sessions, features used, session duration and user ratings to understand engagement and improve the user experience.

> Keep `24_mobile_app_usage.csv` in the same folder as this notebook. If your CSV uses different column names, edit the `COL` map in the setup cell.

## Setup

In [ ]:
import sys
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# COLUMN MAP - change the right-hand side to match your CSV headers
# ------------------------------------------------------------------
FILE = "24_mobile_app_usage.csv"
COL = {
    "session": "Session_ID",
    "user": "User_ID",
    "date": "Session_Date",
    "duration": "Session_Duration_Mins",
    "feature": "Feature_Used",
    "device": "Device_Type",
    "age_group": "Age_Group",      # if absent, script bins a numeric "Age" column
    "rating": "User_Rating",
}

## Task 1: Load and Inspect the Dataset

Load the CSV and inspect shape, column types, sample rows and summary statistics. An age-group column is created from `Age` if it is missing.

In [ ]:
df = pd.read_csv(FILE)
print("=== 1. INSPECTION ===")
print("Shape:", df.shape)
print(df.head(), "\n")
df.info()
print(df.describe(include="all").T, "\n")

missing_cols = [c for k, c in COL.items() if c not in df.columns and k != "age_group"]
if missing_cols:
    raise SystemExit(f"Column(s) not found: {missing_cols}. Update the COL map at the top.\n"
                     f"Available columns: {list(df.columns)}")

# Age group: use existing column, else bin numeric Age
if COL["age_group"] not in df.columns:
    if "Age" in df.columns:
        df[COL["age_group"]] = pd.cut(df["Age"], bins=[0, 17, 24, 34, 44, 54, 120],
                                      labels=["<18", "18-24", "25-34", "35-44", "45-54", "55+"])
    else:
        print("WARNING: no age-group or Age column; task 8 will be skipped.")

=== 1. INSPECTION ===
Shape: (2412, 8)
  Session_ID User_ID         Session_Date  ...  Device_Type Age_Group User_Rating
0    S100333   U1337  2025-02-23 20:04:25  ...          iOS     13-17         5.0
1    S101904   U1241  2025-10-21 19:17:12  ...          Web     13-17         4.0
2    S101884   U1055  2025-10-18 22:27:38  ...      Android     25-34         4.0
3    S102272   U1207  2025-12-09 18:58:36  ...          Web     18-24         5.0
4    S101496   U1360  2025-08-24 08:18:13  ...      Android     25-34         4.0

[5 rows x 8 columns] 

<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Session_ID             2412 non-null   str    
 1   User_ID                2412 non-null   str    
 2   Session_Date           2412 non-null   str    
 3   Session_Duration_Mins  2339 non-null   float64
 4   Feature_Used           2412 non-n

## Task 2: Missing Values and Duplicates

Count nulls per column and duplicate `Session_ID` records. Duplicates are removed, keeping the first occurrence.

In [ ]:
print("\n=== 2. MISSING VALUES & DUPLICATES ===")
print(df.isna().sum())
dup_ids = df.duplicated(subset=COL["session"]).sum()
print(f"Duplicate {COL['session']} records: {dup_ids}")
df = df.drop_duplicates(subset=COL["session"], keep="first").copy()


=== 2. MISSING VALUES & DUPLICATES ===
Session_ID                 0
User_ID                    0
Session_Date               0
Session_Duration_Mins     73
Feature_Used               0
Device_Type                0
Age_Group                  0
User_Rating              110
dtype: int64
Duplicate Session_ID records: 12


## Task 3: Handle Missing Duration and Rating

**Approach:** median imputation within each feature. Session duration is right-skewed, so the median resists extreme values; ratings are ordinal, so a mean could produce invalid scores. Flag columns record which rows were imputed.

In [ ]:
# Justification:
#  - Duration is right-skewed (a few very long sessions), so the MEDIAN is a safer fill
#    than the mean. We use the median within the same feature (behaviour differs by
#    feature), falling back to the overall median.
#  - Non-positive durations are impossible, so they are treated as missing too.
#  - Ratings are ordinal (e.g. 1-5), so the mean would produce non-valid values. We use
#    the per-feature median. Flag columns keep a record of what was imputed.
print("\n=== 3. MISSING VALUE TREATMENT ===")
dur, rat, feat = COL["duration"], COL["rating"], COL["feature"]
df.loc[df[dur] <= 0, dur] = np.nan
df["duration_imputed"] = df[dur].isna()
df["rating_imputed"] = df[rat].isna()
df[dur] = df[dur].fillna(df.groupby(feat)[dur].transform("median")).fillna(df[dur].median())
df[rat] = df[rat].fillna(df.groupby(feat)[rat].transform("median")).fillna(df[rat].median())
print("Imputed durations:", int(df["duration_imputed"].sum()),
      "| Imputed ratings:", int(df["rating_imputed"].sum()))
print("Remaining nulls in key columns:", int(df[[dur, rat]].isna().sum().sum()))


=== 3. MISSING VALUE TREATMENT ===
Imputed durations: 70 | Imputed ratings: 110
Remaining nulls in key columns: 0


## Task 4: Datetime Features

Convert `Session_Date` to datetime and extract month, weekday and hour.

In [ ]:
print("\n=== 4. DATETIME ===")
df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")
bad_dates = df[COL["date"]].isna().sum()
if bad_dates:
    print(f"Dropping {bad_dates} rows with unparseable dates")
    df = df.dropna(subset=[COL["date"]])
df["Month"] = df[COL["date"]].dt.month_name()
df["Weekday"] = df[COL["date"]].dt.day_name()
df["Hour"] = df[COL["date"]].dt.hour
if df["Hour"].nunique() == 1:
    print("NOTE: Session_Date has no time component, so hour analysis (task 10) is not meaningful.")
print(df[[COL["date"], "Month", "Weekday", "Hour"]].head())


=== 4. DATETIME ===
         Session_Date     Month   Weekday  Hour
0 2025-02-23 20:04:25  February    Sunday    20
1 2025-10-21 19:17:12   October   Tuesday    19
2 2025-10-18 22:27:38   October  Saturday    22
3 2025-12-09 18:58:36  December   Tuesday    18
4 2025-08-24 08:18:13    August    Sunday     8


## Task 5: Headline KPIs

Total sessions, unique users and average session duration.

In [ ]:
print("\n=== 5. KPIs ===")
total_sessions = len(df)
unique_users = df[COL["user"]].nunique()
avg_duration = df[dur].mean()
print(f"Total sessions: {total_sessions}")
print(f"Unique users:   {unique_users}")
print(f"Avg duration:   {avg_duration:.2f} mins")
print(f"Sessions per user: {total_sessions / unique_users:.2f}")


=== 5. KPIs ===
Total sessions: 2400
Unique users:   407
Avg duration:   17.62 mins
Sessions per user: 5.90


## Task 6: Top 10 Features

The 10 most frequently used app features.

In [ ]:
print("\n=== 6. TOP 10 FEATURES ===")
top_features = df[feat].value_counts().head(10)
print(top_features)


=== 6. TOP 10 FEATURES ===
Feature_Used
Flashcards            396
Practice Quiz         393
Video Lessons         297
Notes                 268
Mock Test             191
Study Planner         181
Discussion Forum      147
Progress Dashboard    137
Pomodoro Timer        134
Doubt Chat            105
Name: count, dtype: int64


## Task 7: Average Duration by Device

Average session duration compared across device types.

In [ ]:
print("\n=== 7. AVG DURATION BY DEVICE ===")
by_device = df.groupby(COL["device"])[dur].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
print(by_device.round(2))


=== 7. AVG DURATION BY DEVICE ===
              mean  median  count
Device_Type                      
Web          20.59   14.72    182
Tablet       19.11   14.85    182
iOS          18.63   13.60    677
Android      16.52   12.20   1359


## Task 8: Age Group Ranking

Age groups ranked by total sessions.

In [ ]:
print("\n=== 8. AGE GROUPS BY SESSIONS ===")
if COL["age_group"] in df.columns:
    age_rank = df[COL["age_group"]].value_counts().rename_axis("Age_Group").reset_index(name="Sessions")
    age_rank["Rank"] = age_rank["Sessions"].rank(ascending=False, method="min").astype(int)
    print(age_rank)


=== 8. AGE GROUPS BY SESSIONS ===
  Age_Group  Sessions  Rank
0     18-24      1215     1
1     25-34       489     2
2     13-17       282     3
3     35-44       281     4
4       45+       133     5


## Task 9: Average Rating by Feature

Average user rating compared across features.

In [ ]:
print("\n=== 9. AVG RATING BY FEATURE ===")
rating_by_feat = df.groupby(feat)[rat].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(rating_by_feat.round(2))


=== 9. AVG RATING BY FEATURE ===
                    mean  count
Feature_Used                   
Flashcards          4.21    396
Pomodoro Timer      4.09    134
Practice Quiz       4.03    393
Progress Dashboard  3.96    137
Video Lessons       3.91    297
Notes               3.88    268
Mock Test           3.66    191
Doubt Chat          3.66    105
Study Planner       3.65    181
Leaderboard         3.53     96
Discussion Forum    3.46    147
Settings            3.05     55


## Task 10: Peak Hours and Busiest Weekdays

Peak usage hours and busiest weekdays.

In [ ]:
print("\n=== 10. PEAK HOURS & WEEKDAYS ===")
hour_counts = df["Hour"].value_counts().sort_index()
print("Top 5 hours:\n", hour_counts.sort_values(ascending=False).head(5))
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = df["Weekday"].value_counts().reindex(order)
print("\nSessions by weekday:\n", weekday_counts)
print("Busiest weekday:", weekday_counts.idxmax())


=== 10. PEAK HOURS & WEEKDAYS ===
Top 5 hours:
 Hour
20    294
21    259
19    239
18    176
22    164
Name: count, dtype: int64

Sessions by weekday:
 Weekday
Monday       404
Tuesday      376
Wednesday    352
Thursday     375
Friday       307
Saturday     270
Sunday       316
Name: count, dtype: int64
Busiest weekday: Monday


## Task 11: List and Tuple

A **list** holds the most-used features (mutable, ordered); a **tuple** holds the fixed device categories.

In [ ]:
print("\n=== 11. LIST & TUPLE ===")
most_used_features = top_features.index.tolist()          # list: mutable, ordered by usage
DEVICE_CATEGORIES = tuple(sorted(df[COL["device"]].dropna().unique()))  # tuple: fixed set
print("Most-used features (list):", most_used_features)
print("Device categories (tuple):", DEVICE_CATEGORIES)


=== 11. LIST & TUPLE ===
Most-used features (list): ['Flashcards', 'Practice Quiz', 'Video Lessons', 'Notes', 'Mock Test', 'Study Planner', 'Discussion Forum', 'Progress Dashboard', 'Pomodoro Timer', 'Doubt Chat']
Device categories (tuple): ('Android', 'Tablet', 'Web', 'iOS')


## Task 12: Set and Function

A **set** identifies unique users; a function returns the average duration for a chosen feature.

In [ ]:
print("\n=== 12. SET & FUNCTION ===")
unique_user_set = set(df[COL["user"]])
print("Unique users via set:", len(unique_user_set))

def avg_duration_for_feature(data, feature_name):
    """Average session duration (mins) for one feature; None if the feature is not found."""
    subset = data.loc[data[feat] == feature_name, dur]
    return round(subset.mean(), 2) if not subset.empty else None

example = most_used_features[0]
print(f"Avg duration for '{example}':", avg_duration_for_feature(df, example))


=== 12. SET & FUNCTION ===
Unique users via set: 407
Avg duration for 'Flashcards': 13.64


## Task 13: NumPy Statistics

Mean, median and standard deviation of session duration using NumPy.

In [ ]:
print("\n=== 13. NUMPY STATS ===")
d = df[dur].to_numpy()
print(f"Mean:   {np.mean(d):.2f}\nMedian: {np.median(d):.2f}\nStd:    {np.std(d, ddof=1):.2f}")


=== 13. NUMPY STATS ===
Mean:   17.62
Median: 13.00
Std:    15.48


## Task 14: IQR Outliers and Engagement Labels

**Outliers:** sessions above Q3 + 1.5 × IQR.  
**Engagement rule:** Low < Q1, Medium from Q1 to below Q3, High ≥ Q3 of session duration.

In [ ]:
print("\n=== 14. OUTLIERS & ENGAGEMENT ===")
q1, q3 = np.percentile(d, [25, 75])
iqr = q3 - q1
low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["Long_Session_Outlier"] = df[dur] > high_fence
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, upper fence={high_fence:.2f}")
print("Unusually long sessions:", int(df["Long_Session_Outlier"].sum()),
      f"({df['Long_Session_Outlier'].mean():.1%})")

# Rule (documented): based on session-duration quartiles
#   Low    : duration <  Q1
#   Medium : Q1 <= duration < Q3
#   High   : duration >= Q3
def engagement_label(minutes):
    if minutes < q1:
        return "Low"
    if minutes < q3:
        return "Medium"
    return "High"

df["Engagement"] = df[dur].apply(engagement_label)
print(df["Engagement"].value_counts())


=== 14. OUTLIERS & ENGAGEMENT ===
Q1=7.60, Q3=22.40, IQR=14.80, upper fence=44.60
Unusually long sessions: 155 (6.5%)
Engagement
Medium    1200
High       602
Low        598
Name: count, dtype: int64


## Task 15: AppUser Class

A simple `AppUser` class with user ID and device type attributes.

In [ ]:
print("\n=== 15. AppUser CLASS ===")
class AppUser:
    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type
        self.sessions = []                      # list of durations

    def add_session(self, minutes):
        self.sessions.append(minutes)

    def avg_session(self):
        return round(sum(self.sessions) / len(self.sessions), 2) if self.sessions else 0.0

    def __repr__(self):
        return f"AppUser(id={self.user_id}, device={self.device_type}, sessions={len(self.sessions)})"

sample_id = df[COL["user"]].iloc[0]
u = AppUser(sample_id, df.loc[df[COL["user"]] == sample_id, COL["device"]].iloc[0])
for m in df.loc[df[COL["user"]] == sample_id, dur]:
    u.add_session(m)
print(u, "| avg session:", u.avg_session(), "mins")


=== 15. AppUser CLASS ===
AppUser(id=U1337, device=iOS, sessions=7) | avg session: 25.67 mins


## Five Recommendations to Improve Engagement

Generated from the results above. Edit the wording for your report.

In [ ]:
print("\n=== RECOMMENDATIONS ===")
best_rated = rating_by_feat["mean"].idxmax()
worst_rated = rating_by_feat["mean"].idxmin()
long_device = by_device["mean"].idxmax()
short_device = by_device["mean"].idxmin()
peak_hour = int(hour_counts.idxmax())
busiest_day = weekday_counts.idxmax()
recs = [
    f"Promote '{best_rated}' (highest avg rating {rating_by_feat['mean'].max():.2f}) in onboarding and the home screen.",
    f"Investigate '{worst_rated}' (lowest avg rating {rating_by_feat['mean'].min():.2f}) through user feedback and usability testing.",
    f"Sessions on {short_device} average {by_device['mean'].min():.1f} mins versus {by_device['mean'].max():.1f} on {long_device}; "
    f"optimise the {short_device} experience (load time, layout, navigation).",
    f"Schedule notifications, content drops and releases around peak usage ({peak_hour}:00 and {busiest_day}s); "
    f"run maintenance outside those windows.",
    f"Re-engage Low-engagement users ({(df['Engagement'] == 'Low').mean():.0%} of sessions) with personalised prompts to try the "
    f"top features ({', '.join(most_used_features[:3])}), and study the {int(df['Long_Session_Outlier'].sum())} unusually long "
    f"sessions to see if they signal power users or usability problems.",
]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")


=== RECOMMENDATIONS ===
1. Promote 'Flashcards' (highest avg rating 4.21) in onboarding and the home screen.
2. Investigate 'Settings' (lowest avg rating 3.05) through user feedback and usability testing.
3. Sessions on Android average 16.5 mins versus 20.6 on Web; optimise the Android experience (load time, layout, navigation).
4. Schedule notifications, content drops and releases around peak usage (20:00 and Mondays); run maintenance outside those windows.
5. Re-engage Low-engagement users (25% of sessions) with personalised prompts to try the top features (Flashcards, Practice Quiz, Video Lessons), and study the 155 unusually long sessions to see if they signal power users or usability problems.


## Export Deliverables

Saves each deliverable table to the `outputs/` folder and writes the cleaned dataset.

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
top_features.rename_axis("Feature").reset_index(name="Sessions").assign(
    Share_Pct=lambda t: (t["Sessions"] / total_sessions * 100).round(2)
).to_csv("outputs/feature_popularity.csv", index=False)
by_device.round(2).to_csv("outputs/device_comparison.csv")
rating_by_feat.round(2).to_csv("outputs/rating_by_feature.csv")
if COL["age_group"] in df.columns:
    age_rank.to_csv("outputs/age_group_ranking.csv", index=False)
peak = pd.concat([
    hour_counts.rename_axis("Value").reset_index(name="Sessions").assign(Type="Hour"),
    weekday_counts.rename_axis("Value").reset_index(name="Sessions").assign(Type="Weekday"),
])[["Type", "Value", "Sessions"]]
peak.to_csv("outputs/peak_usage_summary.csv", index=False)
df[[COL["session"], COL["user"], dur, "Engagement", "Long_Session_Outlier"]].to_csv(
    "outputs/engagement_labels.csv", index=False)
pd.DataFrame({
    "Metric": ["Total sessions", "Unique users", "Avg duration (mins)", "Mean (NumPy)", "Median (NumPy)",
               "Std dev (NumPy)", "Q1", "Q3", "IQR", "Upper fence", "Outlier sessions"],
    "Value": [total_sessions, unique_users, round(avg_duration, 2), round(np.mean(d), 2),
              round(np.median(d), 2), round(np.std(d, ddof=1), 2), round(q1, 2), round(q3, 2),
              round(iqr, 2), round(high_fence, 2), int(df["Long_Session_Outlier"].sum())],
}).to_csv("outputs/session_statistics.csv", index=False)
with open("outputs/recommendations.txt", "w") as f:
    f.write("\n".join(f"{i}. {r}" for i, r in enumerate(recs, 1)))
print("Saved deliverable tables to the outputs/ folder")

df.to_csv("24_mobile_app_usage_cleaned.csv", index=False)
print("\nSaved cleaned data to 24_mobile_app_usage_cleaned.csv")

Saved deliverable tables to the outputs/ folder

Saved cleaned data to 24_mobile_app_usage_cleaned.csv


## Summary

All deliverables (feature popularity, device comparison, peak-usage summary, engagement labels, session statistics and the five recommendations) are saved in the `outputs/` folder.